# 04 — Evaluation Harness: Paired NKGE vs. Baseline Comparison

This notebook implements the evaluation system described in Section 5 of the design document.

**Primary metric:** Contradiction Rate Improvement % — lower is better for individual scores, higher improvement % shows system value.

**Evaluation design:** Paired — the same player actions run through both NKGE (full system) and Baseline (rolling text summary, no graph/guard/personas) modes. Delta between scores is the headline number.

**Sections:**
1. Config and setup
2. Run paired evaluation (or load existing results)
3. Contradiction analysis
4. Secondary metrics analysis
5. Side-by-side segment comparison
6. Summary report

In [ ]:
import os
import sys
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

sys.path.insert(0, str(Path.cwd().parent))

from langchain_anthropic import ChatAnthropic
from src.graph_client import GraphClient
from src.extraction import ExtractionPipeline
from src.retrieval import CypherRetriever, VectorRetriever
from src.guard import ContradictionGuard
from src.persona import PersonaStore, PersonaGenerator
from src.pipeline import StoryPipeline
from src.schema import SessionState
from src.eval import (
    EvalRunner,
    compute_improvement,
    format_comparison_table,
    load_eval_run,
    SEVERITY_WEIGHTS,
)

print("Imports OK")

## 1. Configuration

In [ ]:
STORY_SEED = "The Iron Tavern"
NUM_SEGMENTS = 20
CHROMA_DIR = "../chroma_store_eval"
EVAL_OUTPUT_DIR = "../eval_runs"

PLAYER_ACTIONS = [
    # Act 1: Confrontation and uneasy alliance (tavern)
    "Kael confronts Maren about the ambush at Thornwood Bridge",
    "Elara arrives at the Iron Tavern with urgent news about bandits on the north road",
    "Kael decides to form an alliance with Maren against the bandit threat",
    "The group sets out from the Iron Tavern toward the Thornwood Bridge",
    "They encounter a patrol of bandits blocking the bridge",
    # Act 2: Journey and encounters
    "Kael uses the Shadow Dagger to intimidate the bandit patrol into retreating",
    "Maren reveals she knows a hidden trail through the Thornwood that bypasses the bridge",
    "Elara discovers an abandoned merchant cart with a map showing the bandit camp location",
    "The group camps for the night near the river and Kael keeps first watch",
    "A wounded survivor from a previous ambush stumbles into their camp seeking help",
    # Act 3: Rising stakes and new locations
    "The survivor tells them the bandit leader has taken prisoners to a cave in the northern hills",
    "Maren scouts ahead and finds the bandit camp is larger than expected with at least twenty fighters",
    "Kael and Elara argue about whether to attempt a rescue or go back to the tavern for reinforcements",
    "Maren offers to sneak into the camp alone using her knowledge of bandit tactics",
    "Elara discovers that the bandit leader is someone from Maren's past who she betrayed years ago",
    # Act 4: Climax and resolution
    "Maren is captured during her infiltration attempt and the bandits send a ransom demand",
    "Kael devises a plan to trade the Shadow Dagger as ransom while Elara frees the prisoners",
    "The bandit leader recognizes the Shadow Dagger and reveals it was originally stolen from him",
    "A fight breaks out and Kael must choose between recovering the dagger and saving Maren",
    "Kael abandons the dagger to save Maren and the group escapes with the freed prisoners back toward the Iron Tavern",
]

gc = GraphClient(os.getenv("NEO4J_URI"), os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD"))
llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.7, max_tokens=1024)
judge_llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.0, max_tokens=2048)

print(f"Story seed: {STORY_SEED}")
print(f"Segments per run: {NUM_SEGMENTS}")
print(f"Severity weights: {SEVERITY_WEIGHTS}")

## 2. Run Paired Evaluation

Set `RUN_FRESH = True` to execute a new paired evaluation, or `False` to load existing results from `eval_runs/`.

In [ ]:
RUN_FRESH = False

if RUN_FRESH:
    persona_store = PersonaStore(persist_dir=CHROMA_DIR)
    if persona_store.count() == 0:
        pg = PersonaGenerator(llm, gc)
        for name in ["Kael", "Maren", "Elara"]:
            persona = pg.generate(name, "main")
            persona_store.upsert_persona(persona)
            print(f"Generated persona for {name}")

    # --- NKGE Run ---
    nkge_pipeline = StoryPipeline(
        graph_client=gc,
        cypher_retriever=CypherRetriever(gc, token_budget=2000),
        vector_retriever=VectorRetriever(persist_dir=CHROMA_DIR),
        extraction=ExtractionPipeline(llm=llm, graph_client=gc),
        llm=llm,
        guard=ContradictionGuard(gc, mode="permissive"),
        persona_store=persona_store,
    )
    nkge_session = SessionState(
        session_id="eval-nkge", story_seed=STORY_SEED,
        current_location="Iron Tavern", present_characters=["Kael", "Maren"],
        last_segment_seq_id=5,
        last_segment_text="The tavern fell silent as Kael and Maren locked eyes.",
        mode="nkge",
    )
    nkge_runner = EvalRunner(gc, nkge_pipeline, judge_llm, output_dir=EVAL_OUTPUT_DIR)
    nkge_result = nkge_runner.run_story(nkge_session, PLAYER_ACTIONS, run_id="nkge_paired_v2")

    # --- Baseline Run ---
    baseline_pipeline = StoryPipeline(
        graph_client=gc,
        cypher_retriever=CypherRetriever(gc, token_budget=2000),
        vector_retriever=VectorRetriever(persist_dir=CHROMA_DIR),
        extraction=ExtractionPipeline(llm=llm, graph_client=gc),
        llm=llm, guard=None, persona_store=None,
    )
    baseline_session = SessionState(
        session_id="eval-baseline", story_seed=STORY_SEED,
        current_location="Iron Tavern", present_characters=["Kael", "Maren"],
        last_segment_seq_id=5,
        last_segment_text="The tavern fell silent as Kael and Maren locked eyes.",
        mode="baseline",
    )
    baseline_runner = EvalRunner(gc, baseline_pipeline, judge_llm, output_dir=EVAL_OUTPUT_DIR)
    baseline_result = baseline_runner.run_story(baseline_session, PLAYER_ACTIONS, run_id="baseline_paired_v2")
    print("Fresh evaluation complete.")
else:
    nkge_result = load_eval_run(f"{EVAL_OUTPUT_DIR}/nkge_paired_v2/eval_output.json")
    baseline_result = load_eval_run(f"{EVAL_OUTPUT_DIR}/baseline_paired_v2/eval_output.json")
    print("Loaded existing evaluation results.")

print(f"NKGE segments: {len(nkge_result.segments)}")
print(f"Baseline segments: {len(baseline_result.segments)}")

## 3. Contradiction Analysis

The primary metric: weighted contradiction score per segment, with severity weights from Section 5.2:
- Critical: 3.0x (dead character alive, destroyed location exists)
- Major: 2.0x (wrong object ownership, impossible location)
- Minor: 1.0x (character visits unvisited place without explanation)
- Soft: 0.5x (character knows something they shouldn't)

In [ ]:
comparison = compute_improvement(nkge_result, baseline_result)

print("=" * 70)
print("HEADLINE: PAIRED EVALUATION RESULTS")
print("=" * 70)
print()
print(format_comparison_table(comparison))
print()
print(f"Contradiction Rate Improvement: {comparison['improvement_pct']:+.1f}%")
print(f"  NKGE mean contradiction score:     {comparison['nkge_contradiction_score']:.2f}")
print(f"  Baseline mean contradiction score:  {comparison['baseline_contradiction_score']:.2f}")

In [ ]:
print("\nContradiction Severity Breakdown:")
print(f"{'Severity':<12} {'NKGE':>6} {'Baseline':>10} {'Delta':>8}")
print("-" * 40)
for sev in ["critical", "major", "minor", "soft"]:
    n = comparison["severity_breakdown"]["nkge"][sev]
    b = comparison["severity_breakdown"]["baseline"][sev]
    delta = n - b
    print(f"{sev:<12} {n:>6} {b:>10} {delta:>+8}")

total_n = comparison["nkge_total_contradictions"]
total_b = comparison["baseline_total_contradictions"]
print("-" * 40)
print(f"{'TOTAL':<12} {total_n:>6} {total_b:>10} {total_n - total_b:>+8}")

## 4. Secondary Metrics Analysis

In [ ]:
print("Per-Segment Coherence Scores:")
print(f"{'Seg':>4} {'NKGE':>6} {'Base':>6}")
print("-" * 20)
for n_seg, b_seg in zip(nkge_result.segments, baseline_result.segments):
    print(f"{n_seg.seq_id:>4} {n_seg.coherence_score:>6.1f} {b_seg.coherence_score:>6.1f}")

print(f"\nMean coherence — NKGE: {comparison['nkge_coherence']:.2f}, Baseline: {comparison['baseline_coherence']:.2f}")
print(f"Coherence delta: {comparison['coherence_delta']:+.2f}")

In [ ]:
print("Per-Segment Retrieval Precision:")
print(f"{'Seg':>4} {'NKGE':>8} {'Base':>8}")
print("-" * 24)
for n_seg, b_seg in zip(nkge_result.segments, baseline_result.segments):
    print(f"{n_seg.seq_id:>4} {n_seg.retrieval_precision:>7.1%} {b_seg.retrieval_precision:>7.1%}")

print(f"\nMean precision — NKGE: {comparison['nkge_retrieval_precision']:.1%}, Baseline: {comparison['baseline_retrieval_precision']:.1%}")

In [ ]:
print("Graph Growth (NKGE only — baseline creates no graph nodes):")
print(f"  Nodes created:         {comparison['nkge_nodes_created']}")
print(f"  Relationships created: {comparison['nkge_rels_created']}")

print("\nPer-Segment Graph Context Tokens:")
print(f"{'Seg':>4} {'Graph':>7} {'Vector':>7}")
print("-" * 22)
for seg in nkge_result.segments:
    print(f"{seg.seq_id:>4} {seg.graph_context_tokens:>7} {seg.vector_context_tokens:>7}")

In [ ]:
print("Guard Violations per Segment (NKGE):")
print(f"{'Seg':>4} {'Count':>6} {'Details'}")
print("-" * 60)
for seg in nkge_result.segments:
    n_violations = len(seg.guard_violations)
    if n_violations == 0:
        print(f"{seg.seq_id:>4} {0:>6} (none)")
    else:
        details = "; ".join(f"{v.check_name}({v.severity})" for v in seg.guard_violations)
        print(f"{seg.seq_id:>4} {n_violations:>6} {details}")

## 5. Side-by-Side Segment Comparison

Compare the highest-contradiction segments from each run to show exactly where the system helped.

In [ ]:
print("=" * 70)
print("WORST SEGMENTS — BASELINE (highest contradiction scores)")
print("=" * 70)
for seg in baseline_result.worst_segments(3):
    print(f"\n--- Segment {seg.seq_id} (score={seg.contradiction_score:.1f}, coherence={seg.coherence_score:.1f}) ---")
    print(f"Action: {seg.player_action}")
    print(f"Text: {seg.generated_text[:300]}..." if len(seg.generated_text) > 300 else f"Text: {seg.generated_text}")
    if seg.contradictions_found:
        print("\nContradictions found:")
        for c in seg.contradictions_found:
            print(f"  [{c.severity}] {c.contradiction_text}")
            print(f"    Conflicts with: {c.conflicting_fact}")
            print(f"    Reasoning: {c.reasoning}")

In [ ]:
print("=" * 70)
print("SAME SEGMENTS — NKGE (for comparison)")
print("=" * 70)
worst_baseline_ids = {s.seq_id for s in baseline_result.worst_segments(3)}
for seg in nkge_result.segments:
    if seg.seq_id in worst_baseline_ids:
        print(f"\n--- Segment {seg.seq_id} (score={seg.contradiction_score:.1f}, coherence={seg.coherence_score:.1f}) ---")
        print(f"Action: {seg.player_action}")
        print(f"Text: {seg.generated_text[:300]}..." if len(seg.generated_text) > 300 else f"Text: {seg.generated_text}")
        if seg.contradictions_found:
            print("Contradictions:")
            for c in seg.contradictions_found:
                print(f"  [{c.severity}] {c.contradiction_text}")
        else:
            print("No contradictions found.")

## 6. Summary Report

Consolidated findings from this evaluation run.

In [ ]:
print("=" * 70)
print("EVALUATION SUMMARY REPORT")
print("=" * 70)
print(f"\nStory seed: {STORY_SEED}")
print(f"Segments per run: {NUM_SEGMENTS}")
print(f"Player actions: {len(PLAYER_ACTIONS)}")
print()
print(format_comparison_table(comparison))
print()
print("KEY FINDINGS:")
print(f"  1. Contradiction Rate Improvement: {comparison['improvement_pct']:+.1f}%")
print(f"     NKGE achieved {comparison['nkge_contradiction_score']:.2f} mean weighted contradiction")
print(f"     score vs. baseline's {comparison['baseline_contradiction_score']:.2f}")
print()
print(f"  2. Coherence Delta: {comparison['coherence_delta']:+.2f}")
if comparison['coherence_delta'] < 0:
    print("     Slight coherence trade-off expected: guard constraints can")
    print("     limit creative freedom in exchange for factual consistency.")
else:
    print("     NKGE maintained or improved coherence alongside consistency.")
print()
print(f"  3. Graph Growth: {comparison['nkge_nodes_created']} nodes, {comparison['nkge_rels_created']} relationships")
print(f"     created during NKGE run (baseline: {comparison['baseline_nodes_created']} nodes)")
print()
print(f"  4. Retrieval Precision: NKGE {comparison['nkge_retrieval_precision']:.1%}")
print(f"     vs. Baseline {comparison['baseline_retrieval_precision']:.1%}")
print()
print("CONCLUSION:")
if comparison['improvement_pct'] > 0:
    print(f"  The NKGE system demonstrates a {comparison['improvement_pct']:.0f}% reduction in")
    print(f"  narrative contradictions compared to the baseline approach.")
    print(f"  The knowledge graph memory provides measurable factual consistency")
    print(f"  improvement with a manageable coherence trade-off of {abs(comparison['coherence_delta']):.2f} points.")
else:
    print("  No improvement detected in this run. Consider increasing segment")
    print("  count or using more complex player actions that stress consistency.")

In [ ]:
gc.close()
print("\nNotebook complete. Evaluation artifacts saved to eval_runs/.")